# Time Series Analysis: Sales / Stock Price Forecasting

## Problem Statement
Forecast future values based on historical time-series data.

## Techniques Used
1. Moving Average
2. ARIMA
3. LSTM (Long Short-Term Memory Neural Networks)

## Dataset
Using `stock_price.csv` containing historical stock data. We will focus on the `last_value` (Close Price) for forecasting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM
import warnings

warnings.filterwarnings('ignore')
plt.style.use('fivethirtyeight')

## 1. Data Loading and Preprocessing

In [ ]:
# Load the dataset
df = pd.read_csv('stock_price.csv')

# Convert date to datetime and set as index
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date')

# Sort by date (ascending) because time series requires chronological order
df = df.sort_index()

# Display first few rows
print(df.head())

# We will focus on 'last_value' which typically represents the closing price
ts_data = df['last_value']

## 2. Time Series Visualization

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(ts_data, label='Stock Price (Last Value)')
plt.title('Historical Stock Prices')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.show()

## 3. Trend, Seasonality, Residual Decomposition

In [ ]:
# Decompose the time series (assuming a period, e.g., 30 for monthly patterns in daily data or just implicit)
# If frequency is irregular, we might need to specify period. Let's assume daily data roughly.
decomposition = seasonal_decompose(ts_data, model='additive', period=30)

fig = decomposition.plot()
fig.set_size_inches(14, 10)
plt.show()

## 4. Stationarity Check (ADF Test)
The Augmented Dickey-Fuller test checks for stationarity. If p-value < 0.05, the series is stationary.

In [ ]:
def test_stationarity(timeseries):
    result = adfuller(timeseries)
    print('ADF Statistic:', result[0])
    print('p-value:', result[1])
    print('Critical Values:')
    for key, value in result[4].items():
        print(f'   {key}: {value}')
    
    if result[1] <= 0.05:
        print("\nData is Stationary")
    else:
        print("\nData is Non-Stationary")

test_stationarity(ts_data)

## 5. Forecasting with Moving Average
A simple moving average can highlight trends and smooth out noise.

In [ ]:
# 30-day Moving Average
ma_30 = ts_data.rolling(window=30).mean()

plt.figure(figsize=(14, 6))
plt.plot(ts_data, label='Original')
plt.plot(ma_30, label='30-Day Moving Average', color='red')
plt.title('Stock Price & Moving Average')
plt.legend()
plt.show()

## 6. Forecasting with ARIMA
AutoRegressive Integrated Moving Average.

In [ ]:
# Split data into train and test
train_size = int(len(ts_data) * 0.8)
train_data, test_data = ts_data[:train_size], ts_data[train_size:]

# Train ARIMA model
# Note: (p, d, q) order needs to be selected. We start with a generic (5, 1, 0) or use auto_arima if available.
# Here we use a simple order for demonstration.
model_arima = ARIMA(train_data, order=(5, 1, 0))
model_fit = model_arima.fit()

# Forecast
forecast_result = model_fit.forecast(steps=len(test_data))
forecast_series = pd.Series(forecast_result, index=test_data.index)

# Plot
plt.figure(figsize=(14, 6))
plt.plot(train_data, label='Training Data')
plt.plot(test_data, label='Actual Data')
plt.plot(forecast_series, label='ARIMA Forecast', color='orange')
plt.title('ARIMA Forecasting')
plt.legend()
plt.show()

# Evaluation
rmse_arima = np.sqrt(mean_squared_error(test_data, forecast_series))
mae_arima = mean_absolute_error(test_data, forecast_series)
print(f'ARIMA RMSE: {rmse_arima}')
print(f'ARIMA MAE: {mae_arima}')

## 7. Forecasting with LSTM (Deep Learning)
Long Short-Term Memory is good for capturing long-term dependencies.

In [ ]:
# Data preparation for LSTM
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(ts_data.values.reshape(-1, 1))

def create_dataset(dataset, time_step=60):
    X, Y = [], []
    for i in range(len(dataset) - time_step - 1):
        a = dataset[i:(i + time_step), 0]
        X.append(a)
        Y.append(dataset[i + time_step, 0])
    return np.array(X), np.array(Y)

# Train/Test split on scaled data
train_size = int(len(scaled_data) * 0.8)
test_size = len(scaled_data) - train_size
train_data_lstm, test_data_lstm = scaled_data[0:train_size, :], scaled_data[train_size:len(scaled_data), :]

time_step = 60
X_train, y_train = create_dataset(train_data_lstm, time_step)
X_test, y_test = create_dataset(test_data_lstm, time_step)

# Reshape input to be [samples, time steps, features]
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# Build LSTM Model
model_lstm = Sequential()
model_lstm.add(LSTM(50, return_sequences=True, input_shape=(100, 1) if False else (time_step, 1)))
model_lstm.add(LSTM(50, return_sequences=False))
model_lstm.add(Dense(25))
model_lstm.add(Dense(1))

model_lstm.compile(optimizer='adam', loss='mean_squared_error')

# Train
model_lstm.fit(X_train, y_train, batch_size=64, epochs=10, verbose=1)

# Prediction
train_predict = model_lstm.predict(X_train)
test_predict = model_lstm.predict(X_test)

# Inverse Transform
train_predict = scaler.inverse_transform(train_predict)
test_predict = scaler.inverse_transform(test_predict)
y_test_original = scaler.inverse_transform(y_test.reshape(-1, 1))

# Evaluation
rmse_lstm = np.sqrt(mean_squared_error(y_test_original, test_predict))
mae_lstm = mean_absolute_error(y_test_original, test_predict)
print(f'LSTM RMSE: {rmse_lstm}')
print(f'LSTM MAE: {mae_lstm}')

In [ ]:
# Plot LSTM Forecast
plt.figure(figsize=(14, 6))

# Shift train predictions for plotting
train_predict_plot = np.empty_like(scaled_data)
train_predict_plot[:, :] = np.nan
train_predict_plot[time_step:len(train_predict)+time_step, :] = train_predict

# Shift test predictions for plotting
test_predict_plot = np.empty_like(scaled_data)
test_predict_plot[:, :] = np.nan
test_predict_plot[len(train_predict)+(time_step*2)+1:len(scaled_data)-1, :] = test_predict

# Plot
plt.plot(scaler.inverse_transform(scaled_data), label='Original Data')
plt.plot(train_predict_plot, label='Train Prediction')
plt.plot(test_predict_plot, label='Test Prediction (LSTM)')
plt.title('LSTM Model Predictions')
plt.legend()
plt.show()

## 8. Conclusion and Comparison
Comparison of RMSE and MAE between ARIMA and LSTM models to determine the better forecasting method for this dataset.